# Reporting Views — Education Analytics
Creates SQL views: executive summary, at-risk students, course pass rates, and faculty performance.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    co.department,
    co.level,
    SUM(co.total_students)      AS total_students,
    SUM(co.active_students)     AS active_students,
    SUM(co.withdrawn_students)  AS withdrawn_students,
    ROUND(AVG(co.cohort_avg_score), 2) AS avg_score,
    ROUND(AVG(co.cohort_pass_rate), 2) AS avg_pass_rate,
    ROUND(AVG(co.retention_rate), 2)   AS avg_retention_rate
FROM gold_cohort_outcomes co
GROUP BY co.department, co.level
""")
print('Created vw_executive_summary')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_at_risk_students AS
SELECT
    ss.student_id,
    ss.programme,
    ss.department,
    ss.level,
    ss.cohort_year,
    ss.gender,
    ss.region,
    ss.avg_score,
    ss.pass_rate,
    ss.total_assessments,
    ss.resit_count,
    ss.retention_risk,
    ss.total_enrolments,
    ss.withdrawn_enrolments
FROM gold_student_scorecards ss
WHERE ss.retention_risk IN ('High', 'Medium')
ORDER BY ss.pass_rate ASC
""")
print('Created vw_at_risk_students')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_course_pass_rates AS
SELECT
    course_id,
    department,
    level,
    assessment_type,
    total_submissions,
    unique_students,
    avg_score,
    pass_rate,
    resit_rate,
    avg_attempts
FROM gold_course_performance
ORDER BY pass_rate ASC
""")
print('Created vw_course_pass_rates')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_faculty_performance AS
SELECT
    fw.faculty_id,
    fw.department,
    fw.role,
    fw.courses_assigned,
    fw.load_band,
    fw.years_at_institution,
    fw.research_active,
    fw.dept_unique_students,
    fw.students_per_course
FROM gold_faculty_workload fw
ORDER BY fw.courses_assigned DESC
""")
print('Created vw_faculty_performance')

In [ ]:
print('All reporting views created.')
for v in ['vw_executive_summary', 'vw_at_risk_students', 'vw_course_pass_rates', 'vw_faculty_performance']:
    cnt = spark.sql(f'SELECT COUNT(*) AS n FROM {v}').collect()[0]['n']
    print(f'  {v}: {cnt} rows')